In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from dataclasses import dataclass
from math import sqrt

FEATURE_TABLE: dict = {
        0: 'group_name',
        1: 'delta_days',
        2: 'nazioni',
        3: 'categorie'
    }


@dataclass
class Config:
    vocabulary: dict | None = None
    dim: int = 64
    # head_size: int = 8
    head_size_div: int = 32
    q_heads: int = head_size_div
    kv_heads: int = q_heads
    context_size: int = 40
    sliding: bool = False
    window_size: int = 4
    MLP_BackBone_expansion: int = 2
    causal: bool = False
    num_experts: int = 12
    routing: int = 2
    num_Block_BackBones: int = 16
    trained_positional: bool = False
    absolute_positional: bool = True
    rotary: bool = False
    n_features: int = 4
    pad_idx: int = 561
    no_sink: bool = True
    use_MoE_BackBone: bool = True
    group_sum: bool = True
    alpha_balance: float = 0.05
    learning_rate: float = 3e-4
    batch_size: int = 256
    training_steps: int = 3100
    sub_emb: bool = True

class AttentionBackBone(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.o_proj = nn.Linear(config.dim, config.dim)
        self.head_size = config.dim // config.head_size_div
        self.qkv = nn.Linear(config.dim, self.head_size * (config.q_heads + 2*config.kv_heads), bias = False)
        self.q_heads   = config.q_heads
        self.kv_heads  = config.kv_heads
        self.context_size = config.context_size
        self.dim = config.dim
        self.causal = config.causal
        self.rotary = config.rotary
        # self.att_large = config.att_large

        event = self.context_size // config.n_features
        trilled_mask = torch.tril(torch.ones(event, event)).repeat_interleave(config.n_features, dim=1).repeat_interleave(config.n_features, dim=0).unsqueeze(0).unsqueeze(1).unsqueeze(2)

        self.register_buffer('tril',trilled_mask)
        self.no_sink = config.no_sink
        if self.no_sink:
            self.W = nn.Linear(config.dim, config.dim)

        self.sliding = config.sliding
        if self.sliding:
            mask = torch.arange(self.context_size).unsqueeze(1) - torch.arange(self.context_size).unsqueeze(0)
            selector = (mask < 0) | (mask > config.window_size)
            window = torch.zeros_like(mask, dtype=torch.float32)
            window[selector] = float('-inf')
            self.register_buffer('sliding_window', window)

        if self.rotary:
            def __compute_angle(T, C):
                i = torch.arange(T).unsqueeze(1)
                k = 1000 ** (torch.arange(C // 2, dtype=torch.float32) / C)
                angle = i / k
                return angle.unsqueeze(0).unsqueeze(1).unsqueeze(2)
            self.register_buffer('angle', __compute_angle(config.context_size, self.head_size))

    def __apply_rotary(self, x, offset):
        T = x.size(3)
        odd = x[:, :, :, :, 0::2]
        even = x[:, :, :, :, 1::2]
        angle = self.angle[:, :, :, offset:offset+T, :]
        x_odd = torch.cos(angle) * odd - torch.sin(angle) * even
        x_even = torch.sin(angle) * odd + torch.cos(angle) * even
        y = torch.empty_like(x)
        y[:, :, :, :, 0::2] = x_odd
        y[:, :, :, :, 1::2] = x_even
        return y

    def forward(self, x, AttentionBackBone_mask=None, use_rotary=False, kv_cache=(None, None), use_cache=False):
        B, T, C = x.size()
        qkv = self.qkv(x)
        if self.q_heads == self.kv_heads:
            q, k, v = qkv.split(self.head_size * self.q_heads, dim=-1)
        else:
            q, kv = qkv.split(self.head_size * self.q_heads, dim=-1)
            k, v = kv.split(self.head_size * self.kv_heads, dim=-1)
        q = q.view(B, T, self.kv_heads, self.q_heads // self.kv_heads, self.head_size)
        k, v = map(lambda x: x.view(B, T, self.kv_heads, 1, self.head_size), (k, v))
        q, k, v = map(lambda x: x.permute(0, 2, 3, 1, 4), (q, k, v))
        offset = 0 if kv_cache[0] is None else kv_cache[0].size(3)

        if use_rotary:
            q, k = map(lambda x: self.__apply_rotary(x, offset), (q, k))

        if use_cache and kv_cache != (None, None):
            k = torch.cat([kv_cache[0], k], dim=3)
            v = torch.cat([kv_cache[1], v], dim=3)
        kv_cache = (k, v)

        attn = q @ k.transpose(3, 4) / sqrt(self.head_size)
        attn = attn.masked_fill((self.tril[:, :, :, :T, :T]==0), float('-inf')) if (self.causal and not use_cache) else attn
        
        if self.sliding:
            attn += self.sliding_window[:T, :T]
        if AttentionBackBone_mask is not None:
            attn = attn + AttentionBackBone_mask

        causal = F.softmax(attn, dim=-1).nan_to_num(nan=0.0)
        y = (causal @ v).permute(0, 3, 1, 2, 4).reshape(B, T, C)

        if self.no_sink:
            y = y * F.sigmoid(self.W(x))
        return self.o_proj(y), kv_cache

class MLP_BackBone(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.up = nn.Linear(config.dim, config.dim * config.MLP_BackBone_expansion)
        self.gate = nn.GELU()
        self.down = nn.Linear(config.dim * config.MLP_BackBone_expansion, config.dim)
        self.MLP_BackBone = nn.Sequential(self.up, self.gate, self.down)
        self.balance_loss = 0
    def forward(self, x):
        return self.MLP_BackBone(x), self.balance_loss

class MoE_BackBone(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_experts = config.num_experts
        self.experts = nn.ModuleList([MLP_BackBone(config) for _ in range(self.num_experts)])
        self.gate = nn.Linear(config.dim, self.num_experts)
        self.k = config.routing

    def forward(self, x):
        B, T, _ = x.size()
        probs = F.softmax(self.gate(x), dim=-1) # B, T, num_experts

        mean_probs = torch.mean(probs.view(B*T, self.num_experts), dim=0)
        topk_probs, topk_indices = probs.topk(self.k, dim=-1)
        topk_probs = topk_probs / topk_probs.sum(dim=-1, keepdim=True)
        y = torch.zeros_like(x)

        frequencies = torch.sum(F.one_hot(topk_indices, self.num_experts), dim=-2).float()
        frequencies = torch.mean(frequencies.view(B*T, self.num_experts), dim=0)

        MoE_BackBone_loss = self.num_experts * torch.sum(frequencies * mean_probs)

        for i in range(self.num_experts):
            o_mask = (topk_indices == i)
            if not o_mask.any(): continue
            mask = o_mask.any(dim=-1, keepdim=True).squeeze(-1)
            out, _ = self.experts[i](x[mask])
            value = topk_probs[o_mask].unsqueeze(1)
            y[mask] += value * out
        return y, MoE_BackBone_loss

class Block_BackBone(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn = AttentionBackBone(config)
        self.use_MoE_BackBone = config.use_MoE_BackBone
        self.ff  = MoE_BackBone(config) if self.use_MoE_BackBone else MLP_BackBone(config)
        self.ln1  = nn.LayerNorm(config.dim)
        self.ln2  = nn.LayerNorm(config.dim)
        
    def forward(self, x, AttentionBackBone_mask=None, use_rotary=False, kv_cache=(None, None), use_cache=False):
        x_attn, kv_cache = self.attn(self.ln1(x), AttentionBackBone_mask, use_rotary, kv_cache, use_cache)
        x = x + x_attn
        ff, balance_loss = self.ff(self.ln2(x))
        x = x + ff
        return x, kv_cache, balance_loss

class (nn.Module):
    def __init__(self, config, vocab_size, sub_vocab=None):
        super().__init__()
        self.decoder: Decoder = Decoder(config=config, vocab_size=vocab_size)
        self.absolute_position = config.absolute_positional
        self.Block_BackBones = nn.ModuleList([Block_BackBone(config) for _ in range(config.num_Block_BackBones_)])
        self.context_size = config.context_size
        self.lm_head = nn.Linear(config.dim, vocab_size)
        self.kv_cache = [(None, None) for _ in range(config.num_Block_BackBones)]
        self.n_features = config.n_features
        self.pad_idx = config.pad_idx
        self.config = config
        self.use_MoE_BackBone = config.use_MoE_BackBone
        self.ln1: nn.LayerNorm = nn.LayerNorm(config.dim)
        self.ln2: nn.LayerNorm = nn.LayerNorm(config.dim)
        self.vocabulary = config.vocabulary
        self.pad_idx  = self.vocabulary['<PAD>']
        self.mask_idx = self.vocabulary['<MASK>']
        self.group_sum = config.group_sum
         

        self.sub_vocab = sub_vocab

        self.sub_linear = nn.ModuleList([nn.Linear(config.dim, len(sub_voc)) for sub_voc in self.sub_vocab])
        
        self.stoi = [{i: k for k, i in v.items()} for v in self.sub_vocab]
        self.apply(self._init_weights)
        print(f"Number of parameters: {self.get_num_params()/1e6:.2f}M")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def __compute_AttentionBackBone_matrix(self, attn_vector):
        key_is_pad = (attn_vector == 0)
        return key_is_pad[:, None, None, None, :].float() * (-1e4)


    def __decode(self, logits, greedy=True):
        chosens = None
        for logit in logits:
            logits_last = logit[:, -1, :]  # (B,V)
            if greedy:
                proj = torch.argmax(logits_last, dim=-1, keepdim=True)
                chosens = torch.cat([chosens, proj], dim=1) if chosens is not None else proj # (B,1)
            else:
                probs = F.softmax(logits_last, dim=-1)
                proj  = torch.multinomial(probs, num_samples=1)
                chosens = torch.cat([chosens, proj], dim=1) if chosens is not None else proj

        return chosens

    def __prefill_or_no_cache(self, x, AttentionBackBone_mask, greedy, use_cache, use_rotary):
        logits, _, _, _ = self(x, AttentionBackBone_mask, use_cache, use_rotary=use_rotary)
        chosens = self.__decode(logits, greedy)

        return chosens


    def __compute_AttentionBackBone_mask(self, x):
        AttentionBackBone_mask = torch.ones(x.size(0), self.context_size).to(x.device)
        AttentionBackBone_mask = torch.where((x==self.pad_idx), 0, 1)
        return AttentionBackBone_mask

    def __compute_generations(self, AttentionBackBone_mask):
        max_generation_events = (self.context_size - torch.min(torch.count_nonzero(AttentionBackBone_mask, dim=1)).item()) // self.n_features + 1
        return max_generation_events

    def __print_chosen(self, toks, bacth_to_unmask):

        print("\n\033[1;36m🧠 Model Prediction\033[0m")
        print("\033[36m" + "─" * 42 + "\033[0m")

        for j, batch in enumerate(bacth_to_unmask):
            print("\033[36m" + f'Sequence {j}' + "\033[0m")
            for elem in batch:
                t = toks[j, elem]
                name = self.feature_table.get(elem, f"feature_{elem}")
                value = self.stoi[elem][t.item()]
                print(f"\033[1;33m{name:<15}\033[0m → \033[1;32m{value}\033[0m")
        print("\033[36m" + "─" * 42 + "\033[0m")


    def __update_input(self, x, next_tokens, to_unmask, seed):
        updated = torch.ones(x.size(0), self.n_features).to(x.device)
        for i, row in enumerate(x):
            index = torch.Tensor(to_unmask[i]).to(x.device)
            if (row==self.mask_idx).any():
                row[(row==self.mask_idx)] = next_tokens[i].gather(0, index.long())
                row[seed[i]+self.n_features:seed[i]+2*self.n_features] = self.mask_idx
                updated[i, :] = row[seed[i]:seed[i]+self.n_features]
        return x, updated


    def __compute_masking_tokens(self, x, seed):
        index = torch.cat([torch.arange(i, i+self.n_features).unsqueeze(0) for i in seed], dim=0).to(x.device)
        flipped = torch.gather(x, 1, index=index)
        masked  = (flipped==self.mask_idx).nonzero()
        count   = torch.bincount(masked[:, 0]).tolist()
        to_unmask  = [i.tolist() for i in masked[:, 1].split(count, 0)]
        return to_unmask

    def __generate(self, x, greedy, use_cache, use_rotary, max_generation_events, seed, AttentionBackBone_mask):
        print(f'MAX GENERATION EVENTS: {max_generation_events}')
        print(torch.count_nonzero(AttentionBackBone_mask, dim=1))
        for m in range(max_generation_events):
            if m == 0:
                to_unmask = self.__compute_masking_tokens(x, seed)
            else:
                to_unmask = [[i for i in range(self.n_features)] for _ in range(x.size(0))]
                AttentionBackBone_mask = self.__compute_AttentionBackBone_mask(x)
            next_tokens = self.__prefill_or_no_cache(x, AttentionBackBone_mask, greedy, use_cache, use_rotary)
            # batch_to_consider = torch.nonzero(~seed[seed<self.context_size]).squeeze(1).tolist()
            self.__print_chosen(next_tokens, to_unmask)
            x, updated_event = self.__update_input(x, next_tokens, to_unmask, seed)
            seed = seed + self.n_features
        for i, k in enumerate(x):
            print('-----------------------')
            print(f'FINAL GENERATED X: {k}')
        return x, updated_event

    @torch.inference_mode()
    def generate(self, x, feature_table:dict=FEATURE_TABLE, greedy=False, use_cache=False, use_rotary=False):
        self.feature_table = feature_table
        AttentionBackBone_mask = self.__compute_AttentionBackBone_mask(x)
        max_generation_events = self.__compute_generations(AttentionBackBone_mask)
        seed = torch.count_nonzero(AttentionBackBone_mask, dim=1) - self.n_features
        if use_cache is False:
            x, _ = self.__generate(x, greedy, use_cache, use_rotary, max_generation_events, seed, AttentionBackBone_mask)

    
    
    def __group_sum(self, logits, AttentionBackBone_mask):
        B, T, C = logits.size()
        logits = logits.masked_fill(~AttentionBackBone_mask.unsqueeze(-1), 0)
        logits = logits.view(B, T // self.n_features, self.n_features, C)
        return torch.sum(logits, dim=2, keepdim=False) # B, n_events=10, C


    def forward(self, x, AttentionBackBone_mask, use_cache=False, use_rotary=False):
        x = self.wte(x)


        if self.sub_emb:
            x_col = self.wte_col(self.col_x)
            x     = x + x_col
        total_MoE_BackBone_loss = 0.0
        for i, h in enumerate(self.Block_BackBones):
            x, self.kv_cache[i], MoE_BackBone_loss = h(x, AttentionBackBone_mask=self.__compute_AttentionBackBone_matrix(AttentionBackBone_mask),
                                    use_rotary=use_rotary, kv_cache=self.kv_cache[i], use_cache=use_cache)
            total_MoE_BackBone_loss += MoE_BackBone_loss

        total_MoE_BackBone_loss /= self.config.num_Block_BackBones
        
        x = self.ln1(x)
        if self.group_sum:
            x = self.__group_sum(x, AttentionBackBone_mask)
            x = self.ln2(x)


        return x


    def get_num_params(self, non_embedding=True):
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            if self.absolute_position: n_params -= self.wpe.numel()
            if self.trained_position: n_params -= self.wpe.weight.numel()
        return n_params

    def estimate_mfu(self, fwdbwd_per_iter, dt):
        N = self.get_num_params()
        cfg = self.config
        L, H, Q, T = cfg.num_Block_BackBones, cfg.q_heads, cfg.head_size, cfg.dim
        flops_per_iter = (6*N + 12*L*H*Q*T) * T * fwdbwd_per_iter
        return (flops_per_iter * (1.0/dt)) / 312e12
